# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook demonstrates step-by-step exploration and processing of the FAIR² clinical dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json
```

The data is structured as record sets (clinical features, hormone levels, genetic variants) and includes metadata and multiple `@id` identifiers for every entity.

In [ ]:
# Install `mlcroissant` library if not already installed
!pip install mlcroissant

## 1. Data Loading
Load structured metadata and tabular records using `mlcroissant`. The Croissant schema describes tables, fields, and columns with unique `@id` identifiers.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Name:", metadata.name)
print("Description:", metadata.description)
print("Published:", metadata.datePublished)
print("Keywords:", getattr(metadata, 'keywords', []))
print("License:", getattr(metadata, 'license', None))

## 2. Data Overview
Review available record sets, fields, and their unique `@id` identifiers (as per Croissant convention).

Let's list all available record sets and, for each, their fields and columns. The `@id` properties are central for referencing each data entity.

In [ ]:
# List all record sets with their @id
record_sets = dataset.record_sets

print("Record sets found:")
for rs in record_sets:
    print(f"- {rs['@id']} (name: {rs.get('name', '')})")
    if 'fields' in rs:
        print("  Fields:")
        for field in rs['fields']:
            print(f"    - {field['@id']} (name: {field.get('name','')}, type: {field.get('dataType','')})")
    elif 'columns' in rs:
        print("  Columns:")
        for col in rs['columns']:
            print(f"    - {col['@id']} (name: {col.get('name','')})")
    print()

### View Example Records
Below, we preview records from the first record set (by `@id`). This helps identify available fields and how to reference them by `@id` for later extraction.

**Note:** Replace `record_set_id` with the actual `@id` for the desired record set.

In [ ]:
# Preview records by record set @id
if record_sets:
    first_record_set_id = record_sets[0]['@id']
    print(f"Records from record set @id = {first_record_set_id}:")
    for i, record in enumerate(dataset.records(record_set=first_record_set_id)):
        print(record)
        if i > 2:
            break
else:
    print("No record sets available.")

## 3. Data Extraction
Load tabular data from each record set into a DataFrame for analysis, referencing the RecordSet `@id` and field/column `@id` identifiers.

**Example:** Extract all available record sets into pandas DataFrames. Each DataFrame is keyed by its RecordSet `@id`.

In [ ]:
# Extract all record sets into DataFrames
dataframes = {}

for rs in record_sets:
    rs_id = rs['@id']
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Extracted record set {rs_id}: columns = {df.columns.tolist()}")
        print(df.head(3))
        print()
    else:
        print(f"No records found for {rs_id}")

# Choose one record set to further analyze (for demonstration)
if dataframes:
    sample_record_set_id = list(dataframes.keys())[0]
    print(f"Sample columns for record set {sample_record_set_id}: {dataframes[sample_record_set_id].columns.tolist()}")
    print(dataframes[sample_record_set_id].head())
else:
    print("No dataframes available.")

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps, referencing all fields by their `@id`.
- Filter records based on numeric thresholds
- Normalize fields
- Group and aggregate by categorical attributes

**Note:** If column names are not `@id`, use field mapping from metadata; always try to reference by `@id`.

In [ ]:
# Choose a numeric field @id (replace as needed)
sample_df = dataframes.get(sample_record_set_id, pd.DataFrame())

# Find a numeric field by @id
numeric_fields = []
for rs in record_sets:
    if rs['@id'] == sample_record_set_id:
        for field in rs.get('fields', []):
            # Use dataType or heuristic to pick
            if field.get('dataType') in ['Float', 'Integer', 'Number']:
                numeric_fields.append(field['@id'])
        for col in rs.get('columns', []):
            # Guess numeric columns
            if 'float' in col.get('name', '').lower() or 'level' in col.get('name','').lower():
                numeric_fields.append(col['@id'])
        break

# Default to first numeric field
numeric_field_id = numeric_fields[0] if numeric_fields else sample_df.columns[0]

print(f"Using numeric field @id: {numeric_field_id}")

# Apply filter, normalization, grouping
threshold = 10
if numeric_field_id in sample_df.columns:
    filtered_df = sample_df[sample_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    
    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Group by a non-numeric field
    non_numeric_fields = [col for col in sample_df.columns if col != numeric_field_id]
    group_field = non_numeric_fields[0] if non_numeric_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
else:
    print(f"Numeric field {numeric_field_id} not found in columns: {sample_df.columns.tolist()}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

**Example:** Histogram and scatterplot of filtered numeric field, using column names that represent `@id` where possible.

In [ ]:
# Visualization: Histogram of the numeric field
if numeric_field_id in sample_df.columns:
    plt.figure(figsize=(6, 4))
    sample_df[numeric_field_id].hist(bins=10)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Scatter between two numeric fields (if available)
    if len(numeric_fields) > 1 and numeric_fields[1] in sample_df.columns:
        plt.figure(figsize=(6, 4))
        plt.scatter(sample_df[numeric_fields[0]], sample_df[numeric_fields[1]])
        plt.xlabel(numeric_fields[0])
        plt.ylabel(numeric_fields[1])
        plt.title(f"Scatter of {numeric_fields[0]} vs {numeric_fields[1]}")
        plt.show()
else:
    print("Visualization skipped: no appropriate numeric field available.")

## 6. Conclusion
This notebook demonstrates structured exploration of the FAIR² clinical dataset using the `mlcroissant` library:
- Loaded metadata, reviewing dataset details
- Listed all record sets and fields using Croissant `@id` identifiers
- Extracted records into DataFrames, referencing `@id` fields
- Applied common EDA: filtering, normalization, grouping
- Visualized data distributions

The dataset, sourced from clinical and genetic records of NC-21OHD-driven infertility cases, is ideal for academic research and case-based genotype-phenotype analysis. All exploration strictly referenced Croissant schema entities by their `@id`.

Further analysis can be tailored to domain interest, including detailed summaries or phenotype stratification.